# Libraries

In [0]:
# Installing new libraries
%pip install -r ../requirements.txt
%restart_python

In [0]:
# Importing required libraries
import numpy as np
import pandas as pd

# Data

## Reading

In [0]:
# Querying subscription table to get Cineclub signups
def get_cineclub():
    df = spark.sql(
        """
        SELECT 
            DATE_SUB(DATE(REPORTING_DATETIME), (DAYOFWEEK(REPORTING_DATETIME) + 1) % 7) AS `week`, 
            SUM(CASE WHEN PLAN_ID = 'monthly1' THEN 1 ELSE 0 END) AS monthly_signups,
            SUM(CASE WHEN PLAN_ID = 'annual1' THEN 1 ELSE 0 END) AS annual_signups,
            SUM(CASE WHEN PLAN_ID = 'gift-plan' THEN 1 ELSE 0 END) AS gift_signups
        FROM CPX_DATAANALYTICS_GOLD.CPX_DW.FACT_SUBSCRIPTION_EVENTS
        WHERE EVENT_TYPE = 'subscription_created'
        AND STATUS = 'active'
        AND REPORTING_DATETIME IS NOT NULL
        AND REPORTING_DATETIME BETWEEN '2023-03-31' AND '2025-04-03'
        GROUP BY `week`
        ORDER BY `week`
        """
    ).toPandas()

    return df

In [0]:
# Querying film table to get releases
def get_releases():
    df = spark.sql(
        """
        SELECT 
            DATE_SUB(RELEASE_DATE, (DAYOFWEEK(RELEASE_DATE) + 1) % 7) AS `week`,
            COUNT(TITLE_ID) AS films_released
        FROM (
            SELECT 
                TITLE_ID, 
                TITLE_NAME,
                MIN(THEATRICAL_RELEASE_DATE) AS RELEASE_DATE
            FROM CPX_DATAANALYTICS_GOLD.CPX_DW.DIM_FILM
            WHERE PERFORMANCE_TYPE = 'Film Presentation'
            AND THEATRICAL_RELEASE_DATE BETWEEN '2023-03-31' AND '2025-04-03'
            AND NOT UPPER(Film_Title) RLIKE 'FAN EVENT|EARLY ACCESS|FAN FIRST'
            GROUP BY TITLE_ID, TITLE_NAME
        ) AS R
        GROUP BY `week`
        ORDER BY `week`
        """
    ).toPandas()

    return df

In [0]:
# Setting raw data path
directory = '/Volumes/training_feature_development/marketing_mix_modelling/data_v2/'

# Reading spend data
sem = pd.read_excel(directory + 'raw_data/SEM EXH & LBE Mar 2023 to Jun 2025 weekly report.xlsx', skiprows=2)
meta = pd.read_excel(directory + 'raw_data/META EXH Mar 2023 to June 2025 weekly report.xlsx')
datorama = pd.read_excel(directory + 'raw_data/Datorama Report All Channels Jan 2023 to Apr 2025 Daily Report.xlsx', skiprows=4, skipfooter=1)
pr = pd.read_csv(directory + 'raw_data/pr.csv')

# Reading other data
attendance_and_revenue = pd.read_csv(directory + 'raw_data/attendance_and_revenue.csv')
cineclub = get_cineclub() # Backup in 'cineclub_signups.csv'
cineplex_web = pd.read_csv(directory + 'raw_data/Cineplex dot com - Day.csv', skiprows=11)
mass_offers = pd.read_csv(directory + 'raw_data/mass_offers.csv')
releases = get_releases() # Backup in 'film_releases.csv'

## Cleaning

In [0]:
# Replacing headers
attendance_and_revenue.columns = ['date', 'attendance', 'exh_revenue', 'box_office_revenue', 'concessions_revenue', 'lbe_revenue']
cineplex_web.columns = ['date', 'website_visits', 'unique_visitors', 'orders']

# Cleaning column names
for df in [sem, meta, datorama, pr, attendance_and_revenue, cineclub, cineplex_web, mass_offers, releases]:
  df.columns = df.columns.str.lower().str.replace(' ', '_')

# Creating new columns
cineclub['cineclub_signups'] = cineclub.filter(like='signups').sum(axis=1)
pr['spend'] = pr[pr.columns[1:-3]].sum(axis=1)
mass_offers['mass_offer'] = mass_offers.max(axis=1, numeric_only=True)

# Dropping observations with no data
meta = meta.dropna(subset='amount_spent_(cad)')

# Making data frequency weekly, starting Friday
sem['week'] = pd.to_datetime(sem['week']).dt.to_period('W-THU').dt.start_time
meta['week'] = pd.to_datetime(meta['reporting_starts']).dt.to_period('W-THU').dt.start_time
datorama['week'] = pd.to_datetime(datorama['day']).dt.to_period('W-THU').dt.start_time
pr['week'] = pd.to_datetime(pr['week']).dt.to_period('W-THU').dt.start_time
attendance_and_revenue['week'] = pd.to_datetime(attendance_and_revenue['date']).dt.to_period('W-THU').dt.start_time
cineclub['week'] = pd.to_datetime(cineclub['week'])
cineplex_web['week'] = pd.to_datetime(cineplex_web['date']).dt.to_period('W-THU').dt.start_time
mass_offers['week'] = pd.to_datetime(mass_offers['week']).dt.to_period('W-THU').dt.start_time
releases['week'] = pd.to_datetime(releases['week'])

# Splitting Datorama dataframe
snapchat = datorama[datorama['publisher'] == 'Snapchat'].copy()
tiktok = datorama[datorama['publisher'] == 'TikTok'].copy()
programmatic = datorama[datorama['channel']!='Social'].copy()

In [0]:
# Renaming SEM LOBs
col = sem['account_name']
sem['lob'] = np.select(
    [col.str.contains('B2B|G&E'), col.str.contains('Event'), col.str.contains('Brand'),
     col.str.contains('Cineclub'), col.str.contains('Film'), col.str.contains('Prepaid'),
     col.str.contains('IMAX'), col.str.contains('EXH - Cineplex|Film Support 1_SEM')],
    ['G&E', 'Event', 'Brand', 'Cineclub', 'Film', 'Prepaid', 'IMAX', 'Exhibition'],
    default='Other'
)

# Renaming Meta LOBs
meta['campaign_short'] = meta['campaign_name'].str.extract(r'(?:CEC|CEA)_(.*?)_')
col = meta['campaign_short']
meta['lob'] = np.select(
    [col=='GRP', col=='A&C', col=='BRA',
     col=='CCB', col=='FLM', col=='PPP',
     col=='IMX', col=='TFB', col=='EXH'],
    ['G&E', 'Event', 'Brand', 'Cineclub', 'Film', 'Prepaid', 'IMAX', 'F&B', 'Exhibition'],
    default='Other'
)

# Renaming Snapchat LOBs
col = snapchat['line_of_business']
snapchat['lob'] = np.select(
    [col=='Brand', col=='Film Marketing', col=='Enhanced Experiences',
     col=='Performance'],
    ['Brand', 'Film', 'IMAX', 'Performance'],
    default='Other'
)

# Renaming TikTok LOBs
col = tiktok['line_of_business']
tiktok['lob'] = np.select(
    [col=='Brand', col=='Film Marketing'],
    ['Brand', 'Film'],
    default='Other'
)

# Renaming Programmatic LOBs
col = programmatic['line_of_business']
programmatic['lob'] = np.select(
    [col=='Brand', col=='Cineclub', col=='Film Marketing', 
     col=='B2C', col=='Enhanced Experiences', col=='Performance'],
    ['Brand', 'Cineclub', 'Film', 'Prepaid', 'IMAX' ,'Performance'],
    default='Other'
)

In [0]:
# Adding SEM objectives
col = sem['campaign']
sem['objective'] = np.select(
    [col.str.contains('Awareness|_AW', na=False), 
     sem['account_name'].str.contains('Awareness', na=False),
     col.str.contains('Consideration|_CONS|_CN', na=False),
     sem['account_name'].str.contains('Consideration|B2B', na=False),
     col.str.contains('Conversion|_CVN|_CVS|001_CP2K790', na=False),
     sem['account_name'].str.contains('Conversion', na=False)],
    ['Awareness', 'Awareness', 'Consideration', 'Consideration', 'Conversion', 'Conversion'],
    default='Unknown')

# Adding Meta objectives
col = meta['result_indicator']
meta['objective'] = np.select(
    [col.str.contains('reach', na=False), 
     col.str.contains('link_click|landing_page_view', na=False),
     col.str.contains('offsite_conversion|subscribe_website', na=False)],
    ['Awareness', 'Consideration', 'Conversion'],
    default='Unknown')

# Changing Datorama objectives
cols = ['Awareness', 'Consideration', 'Conversion']
snapchat['objective'] = np.where(snapchat['objective'].isin(cols), snapchat['objective'], 'Unknown')
tiktok['objective'] = np.where(tiktok['objective'].isin(cols), tiktok['objective'], 'Unknown')
programmatic['objective'] = np.where(programmatic['objective'].isin(cols), programmatic['objective'], 'Unknown')
pr['objective'] = 'Conversion'

In [0]:
# Setting channel column values
sem['channel'] = 'SEM'
meta['channel'] = 'Meta'
snapchat['channel'] = 'Snapchat'
tiktok['channel'] = 'TikTok'
programmatic['channel'] = 'Programmatic'

# Setting PR column values
pr = pr.assign(
    week=pr['week'],
    lob='PR',
    channel='PR',
    campaign_name='N/A',
    spend=pr['spend'],
    impressions=0,
    clicks=0,
    revenue=0
)

In [0]:
# Selecting columns
cols = [
    ['week', 'objective', 'lob', 
     'channel', 'campaign', 'cost', 
     'impr.', 'clicks', 'revenue'],
    ['week', 'objective', 'lob', 
     'channel', 'campaign_name', 'amount_spent_(cad)', 
     'impressions', 'link_clicks', 'purchase_revenue'],
    ['week', 'objective', 'lob', 
     'channel', 'campaign_name', 'investment', 
     'impressions', 'clicks', 'purchase_revenue'],
    ['week', 'objective', 'lob', 
     'channel', 'campaign_name', 'investment', 
     'impressions', 'clicks', 'purchase_revenue'],
    ['week', 'objective', 'lob', 
     'channel', 'campaign_name', 'investment', 
     'impressions', 'clicks', 'purchase_revenue']
    ]

sem = sem[cols[0]]
meta = meta[cols[1]]
snapchat = snapchat[cols[2]]
tiktok = tiktok[cols[3]]
programmatic = programmatic[cols[4]]

# Renaming columns
cols = [
    'week', 'objective', 'lob', 
    'channel', 'campaign_name', 'spend', 
    'impressions', 'clicks', 'revenue'
]

sem.columns = cols
meta.columns = cols
snapchat.columns = cols
tiktok.columns = cols
programmatic.columns = cols

In [0]:
# Combining data
spend_df = pd.concat([sem, meta, snapchat, tiktok, programmatic, pr[cols]]).fillna(0)
response_df = pd.concat([
    attendance_and_revenue.groupby('week').agg({'attendance': 'sum', 'exh_revenue': 'sum'}),
    cineclub.set_index('week')[['cineclub_signups']],
    cineplex_web.groupby('week').agg({'website_visits': 'sum'})
], axis=1).reset_index()
control_df = pd.concat([
    mass_offers.groupby('week').agg({'mass_offer': 'max'}),
    releases.set_index('week')
], axis=1).reset_index().fillna(0)

# Filtering data for trailing 2 years
start_date, end_date = pd.Timestamp('2023-03-31'), pd.Timestamp('2025-03-31')
spend_df = spend_df[spend_df['week'].between(start_date, end_date)]
response_df = response_df[response_df['week'].between(start_date, end_date)]
control_df = control_df[control_df['week'].between(start_date, end_date)]

# Renaming control columns
control_df.columns = ['week', 'mass_offer', 'films_released']

# Feature Engineering

In [0]:
# Defining holidays
holidays_by_name = {
    "New Year's Day": ['2023-01-01', '2024-01-01', '2025-01-01'],
    'Family Day': ['2023-02-20', '2024-02-19', '2025-02-17'],
    'Good Friday': ['2023-04-07', '2024-03-29', '2025-04-18'],
    'Easter Monday': ['2023-04-10', '2024-04-01', '2025-04-21'],
    'Victoria Day': ['2023-05-22', '2024-05-20', '2025-05-19'],
    'Canada Day': ['2023-07-01', '2024-07-01', '2025-07-01'],
    'Labour Day': ['2023-09-04', '2024-09-02', '2025-09-01'],
    'Thanksgiving': ['2023-10-09', '2024-10-14', '2025-10-13'],
    'Christmas Day': ['2023-12-25', '2024-12-25', '2025-12-25']
}

holidays = [pd.to_datetime(date) for date_list in holidays_by_name.values() for date in date_list]
holidays = pd.Series(holidays).dt.to_period('W-THU').dt.start_time

# Creating new columns
control_df['summer'] = np.where(control_df['week'].dt.month.isin([7, 8]), 1, 0)
control_df['holiday'] = np.where(control_df['week'].isin(holidays), 1, 0)

In [0]:
control_df.to_csv(directory + 'model_data/exh_control_data.csv', index=False)
response_df.to_csv(directory + 'model_data/exh_response_data.csv', index=False)
spend_df.to_csv(directory + 'model_data/exh_spend_data.csv', index=False)